In [1]:
import base64
from dotenv import load_dotenv
import os

from anthropic import Anthropic
from anthropic.types import Message

In [2]:
load_dotenv(override=True)

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

client = Anthropic(api_key=anthropic_api_key)

model = "claude-sonnet-4-5-20250929"

In [ ]:
def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)

In [11]:
def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024    # max tokens for thinking when claude generates the response (1024 is min)
    ):
    params = {
        "model": model,
        "max_tokens": 4000,     # must be higher than 'thinking_budget'
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "adaptive",
            "budget_tokens": thinking_budget
        }

    if tools:
        params["tools"] = tools
    
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join(
        [block for block in message.content if block.type == "text"]
    )
    

In [22]:
with open("../../data/documents/earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8") 

messages = []

add_user_message(messages, [
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        }
    },
    {
        "type": "text",
        "text": "How were Earth's atmosphere and oceans formed?"
    }
])

chat(messages)

Message(id='msg_011CfruRdFsKXZTSn4TAkVTz', container=None, content=[TextBlock(citations=None, text="According to the document, Earth's atmosphere and oceans were formed by **volcanic activity and outgassing**.\n\nThe process worked as follows:\n\n1. **Water vapor** from volcanic activity and outgassing condensed into the oceans\n\n2. This was **augmented by water and ice** from asteroids, protoplanets, and comets\n\n3. The document notes that sufficient water to fill the oceans may have been on Earth since it formed\n\n4. **Atmospheric greenhouse gases** kept the oceans from freezing when the newly forming Sun had only 70% of its current luminosity\n\n5. By 3.5 billion years ago (Ga), Earth's magnetic field was established, which helped prevent the atmosphere from being stripped away by solar wind\n\nSo in summary, the atmosphere and oceans originated primarily from volcanic outgassing, with additional water contributions from space objects like asteroids and comets.", type='text')], m